# 特征选择

In [1]:
import numpy as np
import warnings

from sklearn.feature_selection import VarianceThreshold, SelectKBest
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

C:\MySoftware\Anaconda3\envs\cv1\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\MySoftware\Anaconda3\envs\cv1\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
C:\MySoftware\Anaconda3\envs\cv1\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
X = np.array([
    [0, 2, 0, 3],
    [0, 1, 4, 3],
    [1, 1, 1, 3],
    [1, 2, 3, 1],
    [2, 3, 4, 3]
], dtype=np.float32)
Y = np.array([1, 2, 1, 2, 1])

## 方差选择法

In [3]:
# 基于方差选择最优的特征属性，阈值设为0.6，保留方差≥0.6的特征
variance = VarianceThreshold(threshold=0.6)
print(variance)
variance.fit(X)
print("各个特征属性的方差为:")
print(variance.variances_)
print('-----------------')
print(variance.transform(X))

VarianceThreshold(threshold=0.6)
各个特征属性的方差为:
[0.56 0.56 2.64 0.64]
-----------------
[[0. 3.]
 [4. 3.]
 [1. 3.]
 [3. 1.]
 [4. 3.]]


## 相关系数法

In [4]:
# 选择与目标变量最相关的2个特征
sk1 = SelectKBest(f_regression, k=2)
sk1.fit(X, Y)
print(sk1)
print('------------')
# 检验得分，选择得分最高的2个特征
print(sk1.scores_)
print('------------')
print(sk1.transform(X))

SelectKBest(k=2, score_func=<function f_regression at 0x000001D76D8ADC10>)
------------
[0.36 0.36 1.32 1.8 ]
------------
[[0. 3.]
 [4. 3.]
 [1. 3.]
 [3. 1.]
 [4. 3.]]


## 卡方检验

In [5]:
# 使用chi2的时候要求特征属性的取值为非负数，选择2个最佳特征
sk2 = SelectKBest(chi2, k=2)
sk2.fit(X, Y)
print(sk2)
print(sk2.scores_)
print(sk2.transform(X))

SelectKBest(k=2, score_func=<function chi2 at 0x000001D76D8ADAF0>)
[0.375      0.16666667 1.68055556 0.46153846]
[[0. 3.]
 [4. 3.]
 [1. 3.]
 [3. 1.]
 [4. 3.]]


## Wrapper-递归特征消除法

In [6]:
# 基于特征消去法做的特征选择
estimator = LogisticRegression()
# 每次迭代删除2个最不重要的特征，最终选择3个特征
selector = RFE(estimator, step=2, n_features_to_select=3)
selector = selector.fit(X, Y)
print(selector.support_)
print(selector.n_features_)
# 特征排名（1表示被选中）
print(selector.ranking_)
print(selector.transform(X))

[ True False  True  True]
3
[1 2 1 1]
[[0. 0. 3.]
 [0. 4. 3.]
 [1. 1. 3.]
 [1. 3. 1.]
 [2. 4. 3.]]


## Embedded【嵌入法】-基于惩罚项的特征选择法

In [7]:
X2 = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [4.9, 3., 1.4, 0.2],
    [-6.2, 0.4, 5.4, 2.3],
    [-5.9, 0., 5.1, 1.8]
], dtype=np.float64)
Y2 = np.array([0, 0, 2, 2])
# 较小的C值意味着更强的正则化
# estimator = LogisticRegression(penalty='l1', C=0.1, solver='liblinear')
estimator = LogisticRegression(penalty='l2', C=0.1)
# 系数绝对值大于0.09的特征被选中
sfm = SelectFromModel(estimator, threshold=0.09)
sfm.fit(X2, Y2)
print(sfm.transform(X2))
print("系数:")
print(sfm.estimator_.coef_)

[[ 5.1  1.4]
 [ 4.9  1.4]
 [-6.2  5.4]
 [-5.9  5.1]]
系数:
[[-0.28319012 -0.07821437  0.09858841  0.04716025]]
